### TEST · dominio pequeño (30–48 °N, 5 °W–20 °E) · PVAR
Mismo PVAR que producción sobre las 30 PCs `_wmed`. Todo se guarda en `outputs/var_6h_wmed/`; producción no se toca.

In [1]:
import pandas as pd
import numpy as np
import json
from utils.tv_var.harmonic_standardizer import HarmonicStandardizer
from utils.tv_var.periodic_var import PeriodicVAR, select_order

import matplotlib.pyplot as plt

pcs_sel_6h = pd.read_csv('outputs/pca/pcs_msl_30_6h_wmed.csv', index_col=0, parse_dates=True)

#### Quitar media y desviación estacionales (la estacionalidad va aquí, no en los coeficientes)
harm_std_pcs = HarmonicStandardizer(n_harmonics=3)
harm_std_pcs.fit(pcs_sel_6h)
pcs_sel_6h_z = harm_std_pcs.transform(pcs_sel_6h)
print(pcs_sel_6h_z.shape)

(126864, 30)


In [2]:
bic_0b = select_order(pcs_sel_6h_z, p_values=[6, 8, 10, 12, 16], n_harm=0, freq="6h")
bic_0b

,nobs,loglike,n_params,bic,best
p,,,,,
6,126848,-1.946858e+06,11010,4.023092e+06,False
8,126848,-1.933072e+06,12810,4.016670e+06,False
10,126848,-1.917246e+06,14610,4.006170e+06,True
12,126848,-1.912149e+06,16410,4.017127e+06,False
16,126848,-1.899414e+06,20010,4.033961e+06,False


In [3]:
var_h0 = PeriodicVAR(p=10, n_harm=0, freq="6h").fit(pcs_sel_6h_z)
var_h1 = PeriodicVAR(p=10, n_harm=1, freq="6h").fit(pcs_sel_6h_z)
var_h2 = PeriodicVAR(p=10, n_harm=2, freq="6h").fit(pcs_sel_6h_z)

pd.DataFrame({"n_harm=0": [var_h0.n_params(), var_h0.loglike(), var_h0.bic()],
              "n_harm=1": [var_h1.n_params(), var_h1.loglike(), var_h1.bic()],
              "n_harm=2": [var_h2.n_params(), var_h2.loglike(), var_h2.bic()]},
             index=["n_params", "loglike", "bic"])

,n_harm=0,n_harm=1,n_harm=2
n_params,1.461000e+04,3.267000e+04,5.073000e+04
loglike,-1.917347e+06,-1.795356e+06,-1.761767e+06
bic,4.006372e+06,3.974610e+06,4.119651e+06


In [4]:
for n, v in {"h0": var_h0, "h1": var_h1, "h2": var_h2}.items():
    print(n, v.stability())

h0 {'rho': 0.04672212737836443, 'rho_daily': 0.9916418718401774, 'rho_step': 0.9979038866279908, 'stable': True}
h1 {'rho': 0.0237695328885974, 'rho_daily': 0.9898075028761876, 'rho_step': 0.9974420779972804, 'stable': True}
h2 {'rho': 0.02765797597605177, 'rho_daily': 0.9902184518827979, 'rho_step': 0.9975455915661394, 'stable': True}


In [5]:
print(var_h2.summary())

                         Periodic VAR                         
Variables (k)                   30    msl_PC1, msl_PC2, msl_PC3, msl_PC4, msl_PC5, msl_PC6, msl_PC7, msl_PC8, msl_PC9, msl_PC10, msl_PC11, msl_PC12, msl_PC13, msl_PC14, msl_PC15, msl_PC16, msl_PC17, msl_PC18, msl_PC19, msl_PC20, msl_PC21, msl_PC22, msl_PC23, msl_PC24, msl_PC25, msl_PC26, msl_PC27, msl_PC28, msl_PC29, msl_PC30
Frequency                       6h    0 days 06:00:00
Exogenous (q)                    0    -
Lags (p)                        10
Harmonics (n_harm)               2
Noise                       gaussian / monthly
Sample                    1940-01-03 12:00 → 2026-10-31 18:00
Observations (nobs)         126854
--------------------------------------------------------------
Coefficients                 45150    (1505 per equation)
Covariance parameters         5580    (12 block(s))
Total parameters             50730
Log-likelihood               -1761766.55
BIC                           4119650.78
-----------

#### Variantes del PVAR y simulaciones
Cada variante tiene un nombre corto que se usa en todo lo demás: `var_{nombre}.pkl`, `simulaciones_{nombre}/` (aquí), `grad_rec/simulaciones_{nombre}_wmed/` (03) y `SIMS` (04).
- `h0`, `h1`, `h2`: número de armónicos en los coeficientes, choques gaussianos.
- sufijo `b`: choques remuestreados de los residuos del mes (bootstrap).

Para probar otra variante: añádela a `VARIANTES` y ponla en `CORRER`. Las simulaciones que ya existen se saltan.

In [ ]:
import os, pickle, time

out = "outputs/var_6h_wmed"
os.makedirs(out, exist_ok=True)
pickle.dump(harm_std_pcs, open(f"{out}/harm_std_pcs.pkl", "wb"))       # común a todas las variantes

burn = int(pd.Timedelta(days=3*365) / pd.Timedelta("6h"))
fechas = pd.date_range("1940-01-01", "2026-12-31 18:00", freq="6h")

VARIANTES = {                              # nombre: (n_harm, choques)
    "h0":  (0, "gaussian"),
    "h1":  (1, "gaussian"),
    "h2":  (2, "gaussian"),
    "h1b": (1, "bootstrap"),
    "h2b": (2, "bootstrap"),
}
CORRER = ["h1b", "h2b"]                    # variantes a ajustar y simular ahora
N_SIMS = 20

In [ ]:
def ruido_bootstrap(var, fechas, seed):
    """Choques remuestreados del pool de residuos de cada mes, sorteados todos de una vez."""
    rng = np.random.default_rng(seed)
    eps = np.empty((len(fechas), var.k_))
    for m, pool in var.resid_month_.items():
        sel = fechas.month == m
        eps[sel] = pool[rng.integers(len(pool), size=sel.sum())]
    return pd.DataFrame(eps, index=fechas, columns=var.columns_)

def ajustar_y_simular(nombre, n_sims=N_SIMS):
    n_harm, choques = VARIANTES[nombre]
    var = PeriodicVAR(p=10, n_harm=n_harm, freq="6h", noise_draw=choques).fit(pcs_sel_6h_z)
    pickle.dump(var, open(f"{out}/var_{nombre}.pkl", "wb"))

    sim_dir = f"{out}/simulaciones_{nombre}"
    os.makedirs(sim_dir, exist_ok=True)
    for i in range(n_sims):
        f_sim = f"{sim_dir}/sim_{i:03d}.parquet"
        if os.path.exists(f_sim):
            continue
        t = time.time()
        if choques == "bootstrap":
            sim_z = var.simulate(noise=ruido_bootstrap(var, fechas, seed=i), burn=burn)
        else:
            sim_z = var.simulate(n_years=87, start="1940-01-01", seed=i, burn=burn)
        harm_std_pcs.inverse_transform(sim_z).astype("float32").to_parquet(f_sim)
        print(f"{nombre} sim {i:03d}  ({time.time() - t:.0f} s)")
    return var

modelos = {nombre: ajustar_y_simular(nombre) for nombre in CORRER}